# Payroll Data Cleaning

## importing dependency and data

In [0]:
import pyspark.sql.functions as F
from pyspark.sql.functions import *


In [0]:
df_payroll_raw = spark.table("bronze.azure_blob_storage.payroll")

In [0]:
df_payroll_raw.display()

## Removing columns inserted by fivetron

In [0]:
df_payroll=df_payroll_raw.drop('_file','_line','_modified','_fivetran_synced')

In [0]:
df_payroll.display()


## Type Casting


#### converting to date

In [0]:

df_payroll = df_payroll.withColumn("pay_period_start", to_date(col("pay_period_start"), "dd-MM-yyyy HH:mm")) \
    .withColumn("pay_period_end", to_date(col("pay_period_end"), "dd-MM-yyyy HH:mm"))\
    .withColumn("pay_date", to_date(col("pay_date"), "dd-MM-yyyy HH:mm")) 


In [0]:
df_payroll.display()

#### Converting to int

In [0]:
df_payroll.dtypes

In [0]:
for col_name, dtype in df_payroll.dtypes:
    if dtype == 'bigint':
        print(col_name,dtype)
        df_payroll = df_payroll.withColumn(col_name, col(col_name).cast('int'))

In [0]:
df_payroll.display()

## Handling Duplicates

In [0]:
if df_payroll.count() > df_payroll.dropDuplicates().count():
    df_payroll = df_payroll.dropDuplicates()
df_payroll.display()

## Handling null values

In [0]:
for column in df_payroll.columns:
    null_count = df_payroll.filter(F.col(column).isNull()).count()
    print(f"{column}: {null_count}")

## writing to silver

In [0]:
df_payroll.write.mode("overwrite").saveAsTable("silver.transformation.payroll")